### 1.2 Формулировка

**Входы**

- Задание и результаты первого знакомства с данными

**Выходы**

- Что предсказываем, по каким данным и для кого
- Требования к данным и критерий оценивания

**Что делаем**

Собираем постановку задачи: что подаём модели, что хотим получить и как поймём, насколько хорошо она работает

In [ ]:
# dev-only: подготовка рабочей папки для интерактивного запуска
from pathlib import Path as _NotebookPath
import os as _notebook_os
_notebook_root = next(
    path for path in (_NotebookPath.cwd(), *_NotebookPath.cwd().parents)
    if (path / 'config/build/project.json').is_file()
)
_notebook_os.chdir(_notebook_root)


##### Для кого строим прогноз

#### Как будем понимать задачу

Предположим, что заёмщик уже подал заявку на кредит Кредитор оценил её, назначил грейд `loan_grade` и предложил ставку `loan_int_rate`

Наша модель нужна инвестору, который решает, финансировать ли такой кредит Поэтому грейд и ставку можно использовать: в выбранный момент они уже известны Возможно, при своей оценке кредитор учёл больше сведений, чем есть у нас Это допущение о сценарии работы, а не установленный факт об источнике данных

Столбец `loan_status` показывает фактический исход: 0 — без дефолта, 1 — дефолт Будем предсказывать его по остальным столбцам Столбец, который предсказываем, называют **целевой переменной**

Это **бинарная классификация**: модель выбирает между двумя классами, то есть двумя вариантами исхода Сначала она выдаёт оценку риска, а затем по выбранной границе — **порогу** — переводим её в класс 0 или 1

#### Что подаём модели и что получаем

Разметка — это известные исходы прошлых кредитов в столбце `loan_status` По ним модель учится отличать дефолты от остальных случаев

| Что определяем | В нашем случае |
| --- | --- |
| Одна запись | Заявка на кредит |
| Вход | Все 11 признаков, кроме `loan_status` |
| Результат | Оценка риска, а после применения порога — класс 0 или 1 |
| Кто использует прогноз | Инвестор, который рассматривает финансирование кредита |
| Когда строим прогноз | Грейд и ставка уже назначены, фактический исход ещё неизвестен |

Алгоритм пока не выбираем: сначала разберём данные

#### Какие данные нужны

Для обучения нужен известный исход кредита — `loan_status` Сам исход не подаём модели как признак

Для новых заявок нужны те же 11 полей и те же единицы измерения Например, ставка записана в процентах, а доля кредита от дохода — числом от 0 до 1

Используем только то, что известно в момент прогноза В нашем сценарии грейд и ставка уже есть, а фактического исхода кредита ещё нет Прошлые дефолты должны относиться именно к прошлому, а не к будущим событиям

Пропуски, повторы и подозрительные значения проверим при EDA Период, за который автор определял дефолт, не указан: модель будет предсказывать метку в том виде, в котором она дана в источнике

#### Как будем оценивать модель

Метрика — численная оценка качества Дефолт считаем положительным исходом: `loan_status = 1`

Для подсчёта ошибок используем четыре обозначения Каждое означает число кредитов

| Обозначение | Что было на самом деле | Что предсказала модель |
| --- | --- | --- |
| **TP** — верно найденные дефолты | Дефолт | Дефолт |
| **FP** — ложные тревоги | Без дефолта | Дефолт |
| **FN** — пропущенные дефолты | Дефолт | Без дефолта |
| **TN** — верно распознанные кредиты без дефолта | Без дефолта | Без дефолта |

**Precision, или точность** — доля настоящих дефолтов среди отмеченных моделью

$$\mathrm{Precision}=\frac{TP}{TP+FP}$$

**Recall, или полнота** — доля найденных дефолтов среди всех настоящих

$$\mathrm{Recall}=\frac{TP}{TP+FN}$$

##### Как получается Average Precision (AP)

Модель выдаёт оценку риска для каждого кредита Порог определяет, какие из них мы считаем дефолтными Снижаем порог — в эту группу попадает больше кредитов На каждом шаге пересчитываем TP, FP и FN, а по ним — precision и recall

**AP — среднее значение precision, где вес каждого шага равен тому, насколько на этом шаге вырос recall**

$$AP=\sum_i (R_i-R_{i-1})P_i,\qquad P_i=\frac{TP_i}{TP_i+FP_i},\quad R_i=\frac{TP_i}{TP_i+FN_i}$$

Здесь $i$ — шаг снижения порога, $P_i$ — precision, $R_i$ — recall, а $R_0=0$

Разберём пример Пусть пять кредитов с разными оценками риска расположены от наибольшего риска к наименьшему Их фактические исходы: **дефолт, без дефолта, дефолт, без дефолта, дефолт**

Снижая порог, будем включать всё больше кредитов Посмотрим на шаги, когда нашли очередной настоящий дефолт

| Модель отметила первые кредиты | TP | FP | Precision | Recall |
| --- | --- | --- | --- | --- |
| 1 кредит | 1 | 0 | $1$ | $1/3$ |
| 3 кредита | 2 | 1 | $2/3$ | $2/3$ |
| Все 5 кредитов | 3 | 2 | $3/5$ | $1$ |

Каждый из трёх дефолтов увеличил recall на $1/3$, поэтому

$$AP=\frac{1}{3}\cdot1+\frac{1}{3}\cdot\frac{2}{3}+\frac{1}{3}\cdot\frac{3}{5}\approx0{,}756$$

На шагах со вторым и четвёртым кредитом recall не вырос — новых дефолтов не нашли Эти шаги не добавляют слагаемых в AP, но ложные тревоги увеличивают FP и уменьшают precision на следующих шагах

Если бы все три дефолта оказались в первых трёх строках, при их добавлении precision каждый раз была бы равна 1 и AP получилась бы равной 1 AP лежит от 0 до 1 Чем выше AP на одной и той же выборке, тем лучше модель располагает дефолты по оценке риска

AP используем для сравнения моделей Конкретный рабочий порог выберем отдельно

Для выбора порога используем **F2**, которая объединяет точность и полноту с большим весом полноты В тех же обозначениях

$$F_2=\frac{5TP}{5TP+FP+4FN}$$

Так мы отдаём приоритет обнаружению дефолтов Стоимости двух видов ошибок не заданы, поэтому финансовую выгоду этим критерием не оцениваем

| Что выбираем | По какому правилу |
| --- | --- |
| Модель | Наибольшая AP на validation |
| Порог | Наибольшая F2 на validation для выбранной модели |
| Итоговое качество | Проверка на test; дополнительные метрики и ошибки разберём в этапе 6 |

Для сравнения возьмём и **базовую модель Dummy**: она не использует признаки, а всем заявкам присваивает риск, равный доле дефолтов в train Минимальную требуемую метрику задание не задаёт